# Implementasi Klasifikasi Konteks False Friend
Tugas: Implementasi sistem klasifikasi NLP dengan Tuning & Antarmuka Interaktif.

## 1. Reshaping Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load dataset
df = pd.read_csv('../data/Dataset Group E - False Friend.csv')

# Reshaping: Gabungkan Sentence_Clean_Indo dan Sentence_Clean_Malay
df_indo = df[['Sentence_Clean_Indo']].rename(columns={'Sentence_Clean_Indo': 'text'})
df_indo['label'] = 'Indonesia'

df_malay = df[['Sentence_Clean_Malay']].rename(columns={'Sentence_Clean_Malay': 'text'})
df_malay['label'] = 'Malaysia'

df_final = pd.concat([df_indo, df_malay], ignore_index=True)
df_final = df_final.dropna(subset=['text'])

print(f"Total data setelah cleaning: {len(df_final)}")

## 2. Training Baseline Model (Naive Bayes)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_final['text'], df_final['label'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred = nb_model.predict(X_test_tfidf)
print(f"Accuracy (NB): {accuracy_score(y_test, y_pred)}")

## 3. Optimasi Model (Random Forest with Tuning)

In [ ]:
# Tuning Random Forest dengan GridSearchCV
param_grid = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train_tfidf, y_train)

best_rf = grid_search.best_estimator_
print(f"Best Params: {grid_search.best_params_}")

y_pred_rf = best_rf.predict(X_test_tfidf)
print(f"Accuracy (RF): {accuracy_score(y_test, y_pred_rf)}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Indonesia', 'Malaysia'], yticklabels=['Indonesia', 'Malaysia'])
plt.title('Confusion Matrix - Optimized Random Forest')
plt.show()

## 4. Antarmuka Interaktif

In [ ]:
def predict_context(kalimat):
    tfidf_vec = vectorizer.transform([kalimat])
    return best_rf.predict(tfidf_vec)[0]

def on_button_clicked(b):
    result = predict_context(text_input.value)
    output_label.value = f"Prediksi: <b>{result}</b>"

text_input = widgets.Text(placeholder='Masukkan kalimat di sini...', description='Kalimat:')
button = widgets.Button(description="Deteksi Konteks", button_style='success')
output_label = widgets.HTML()

button.on_click(on_button_clicked)
display(text_input, button, output_label)